**Predictions of CT, predictions of the bromate trained model and graphs**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit
from sklearn.exceptions import NotFittedError
import torch
import torch.nn as nn
import torch.optim as optim

def filter_by_datetime(ozoneprint, start_date, start_hour, end_date, end_hour,chambnumb):
    # Ensure Date column is datetime
    bromateprint['Date'] = pd.to_datetime(ozoneprint['Date'])

    # Create full datetime column (combine Date + hour)
    bromateprint['DateTime'] = bromateprint['Date']# + pd.to_timedelta(ozoneprint['hour'], unit='h')

    # Build start and end datetime
    start_dt = pd.to_datetime(start_date)# + pd.to_timedelta(start_hour, unit='h')
    end_dt = pd.to_datetime(end_date)# + pd.to_timedelta(end_hour, unit='h')

    mask = (
        (bromateprint['DateTime'] >= start_dt) &
        (bromateprint['DateTime'] <= end_dt) &
        (bromateprint['hour'] >= start_hour) &
        (bromateprint['hour'] <= end_hour) &
        (bromateprint['Chamber'] == chambnumb)
    )
    return ozoneprint[mask].copy()

def bromateplots(bromateprint):
    # --- Step 1: Select dates ---
    start_date=input("Set the start date in the format of 2024-10-01):")
    #start_date="2024-10-01"
    start_hour=int(input("Set the start hour (0-23):"))
    #start_hour=12
    end_date=input("Set the end date in the format of 2024-10-01):")
    #end_date="2024-10-20"
    end_hour=int(input("Set the end hour (0-23):"))
    #end_hour=22
    chambnumb=int(input("Set the chamber (1-4):"))
    filtered_df = filter_by_datetime(bromateprint, start_date, start_hour, end_date, end_hour,chambnumb)

    if filtered_df.empty:
        print("No data in the specified date/hour range.")
        return filtered_df,None

    scaler_features = ["month","Chamber","WaterTemperature","RFTurbidity","FlowRate","Ozonedosage","Residencetime", 'WaterTemperature_1','RFturbidity_1', 'FlowRate_1','Ozonedosage_1','CT_All']
    X_full = filtered_df[scaler_features].copy()

    # --- Step 2: Load ensemble model and predict results ---
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/Bromate best model/"
    modelname="adaboost_vs_12_42.pkl"
    featurescaler_path = os.path.join(save_dir, "feature_scaler.pkl")
    targetscaler_path = os.path.join(save_dir, "target_scaler.pkl")
    model_path = os.path.join(save_dir, modelname)
    #ada_model = joblib.load(model_path)
    #feature_scaler = joblib.load(featurescaler_path)
    #target_scaler = joblib.load(targetscaler_path)

    # --- Step 3: Load Model and Scalers ---
    try:
       ada_model = joblib.load(model_path)
       feature_scaler = joblib.load(featurescaler_path)
       target_scaler = joblib.load(targetscaler_path)
       print("✅ All models and scalers loaded successfully.")
    except FileNotFoundError as e:
        print(f"❌ File not found: {e}")
        return None, None
    except Exception as e:
        print(f"⚠️ Error loading: {e}")
        target_scaler = None # Fallback

    # --- Step 3: Base Prediction ---
    X_base = filtered_df[scaler_features].copy()
    y_pred_base_scaled = ada_model.predict(feature_scaler.transform(X_base))
    y_pred_base = target_scaler.inverse_transform(y_pred_base_scaled.reshape(-1, 1)).ravel()

    # --- Step 4: Sensitivity Input ---
    print("\n--- Bromate Sensitivity Analysis ---")
    change_pct = float(input("Enter Ozonedosage change % (e.g., 20 or -15): "))
    change_pct_1 = float(input("Enter 1 hour timelagged Ozonedosage change % (e.g., 20 or -15): "))
    multiplier = 1 + (change_pct / 100)
    multiplier_1 = 1 + (change_pct_1 / 100)
    print("Recalculating CT for new dosage...")
    # Map the features required by your CT model here:
    X_sens = X_base.copy()
    ct_features = ['hour', 'month', 'Chamber', 'WaterTemperature', 'RFTurbidity', 'FlowRate', 'Ozonedosage', 'Residencetime']
    X_for_ct = filtered_df[ct_features].copy()
    X_for_ct['Ozonedosage'] = X_for_ct['Ozonedosage'] * multiplier
    # --- Step 5: Perturb Features & Re-predict CT ---

    # Update CT_All in the bromate feature set
    X_sens['Ozonedosage'] = X_sens['Ozonedosage'] * multiplier
    X_sens['Ozonedosage_1'] = X_sens['Ozonedosage_1'] * multiplier_1
    X_sens['CT_All'] = ctpredict(X_for_ct)

    # --- Step 6: Predict Bromate with new conditions ---
    y_pred_sens_scaled = ada_model.predict(feature_scaler.transform(X_sens))
    y_pred_sens = target_scaler.inverse_transform(y_pred_sens_scaled.reshape(-1, 1)).ravel()

    # --- Step 7: Plot and compare results ---
    filtered_df['Bromate_limit'] = 5
    plt.figure(figsize=(16,8))
    plt.plot(filtered_df['DateTime'], filtered_df['Bromate_limit'], label="Waternet regulatory limit",color='orange',linestyle=':', linewidth=2)
    plt.plot(filtered_df['DateTime'], y_pred, label="Predicted Bromate concentration", marker='x')
    #plt.scatter(filtered_df['DateTime'], filtered_df['Bromate_WTP_outlet'],color='red', s=100, zorder=5, label="Measured Bromate")
    plt.plot(filtered_df['DateTime'], y_pred_sens,
             label=f"Predicted Bromate (Ozone Dosage {change_pct:+}%)",
             marker='x', color='green', linestyle='--')
    plt.xlabel("DateTime")
    plt.ylabel("Bromate Concentration (μg BrO₃/L * min)")
    plt.title(f"Bromate Predictions vs Bromate regulatory limit in chamber{chambnumb}\n{start_date} {start_hour}:00 to {end_date} {end_hour}:00")
    plt.legend()
    plt.grid(True,alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return filtered_df, y_pred



def ctpredict(X_missing):

    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/OZONE BEST TRAINED MODELS/New dataset/Mix features"
    #modelname="weightedensemble_model.pkl"
    modelname="GB_test_5_v2_42.pkl"
    featurescaler_path = os.path.join(save_dir, "feature_scaler.pkl")
    targetscaler_path = os.path.join(save_dir, "target_scaler.pkl")
    model_path = os.path.join(save_dir, modelname)
    #ensemble_model = joblib.load(model_path)
    gb_model = joblib.load(model_path)
    feature_scaler = joblib.load(featurescaler_path)
    target_scaler = joblib.load(targetscaler_path)
    #method = ensemble_model.get("method")
    #print("The ensemble model method is: ", method)
    model_features = ["Chamber", "WaterTemperature", "RFTurbidity", "FlowRate", "Ozonedosage"]

    #base_models = ensemble_model.get("models", [])
    #scaler_features = ["hour","month","Chamber","WaterTemperature","RFTurbidity","FlowRate","Ozonedosage","Residencetime"]

    def safe_predict(m, f, df):
        X_sub = df.reindex(columns=f, fill_value=0)
        scaler_features = feature_scaler.feature_names_in_ if hasattr(feature_scaler, "feature_names_in_") else f
        X_full = pd.DataFrame(0, index=df.index, columns=scaler_features)
        common_features = [c for c in scaler_features if c in X_sub.columns]
        X_full[common_features] = X_sub[common_features]
        X_scaled = feature_scaler.transform(X_full)
        model_feature_indices = [list(scaler_features).index(c) for c in f if c in scaler_features]
        X_model = X_scaled[:, model_feature_indices]
        #X_subscaled = feature_scaler.transform(X_sub)
        y_predscaled = m.predict(X_model)
        y_pred = target_scaler.inverse_transform(y_predscaled.reshape(-1, 1))
        y_pred = np.array(y_pred).reshape(-1)  # flatten in case shape is (n,1)
        return y_pred

    def predict_simple(m, f, df):
        # 1. Align features with the scaler first (scaler expects 8 features usually)
        scaler_features = list(feature_scaler.feature_names_in_) if hasattr(feature_scaler, "feature_names_in_") else f
        X_full = pd.DataFrame(0, index=df.index, columns=scaler_features)

        # 2. Fill the columns the model actually uses from filtered_df
        for col in f:
            if col in df.columns:
                X_full[col] = df[col]

        # 3. Scale the full feature set
        X_scaled = feature_scaler.transform(X_full)

        # 4. Filter only the features the model was trained on
        # This matches the indices of the model_features within the scaler_features
        model_feature_indices = [scaler_features.index(c) for c in f]
        X_model = X_scaled[:, model_feature_indices]

        # 5. Predict and Inverse Transform
        y_pred_scaled = m.predict(X_model)
        y_pred_unscaled = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        return y_pred_unscaled

    print(f"Predicting using simple model: {modelname}")
    y_pred = predict_simple(gb_model, model_features, X_missing)


    #if method in ["stacking"]:
    #    meta_model = ensemble_model["meta_model"]
    #    test_preds = np.column_stack([safe_predict(m, f, X_missing) for m, f in base_models])
    #    y_pred = meta_model.predict(test_preds)

    #elif method in ["bagging"]:
    #    n_bags = ensemble_model.get("n_bags", 1)
    #    bag_preds = np.zeros((len(X_missing), len(base_models) * n_bags))
    #    idx = 0
    #    for m, features in base_models:
    #        for b in range(n_bags):
    #            bag_preds[:, idx] = safe_predict(m, features, X_missing)
    #            idx += 1
    #    y_pred = np.mean(bag_preds, axis=1)

    #elif method in ["weighted", "optimised", "multi_optimised"]:
    #    weights = np.array(ensemble_model.get("weights"))
    #    test_preds = np.column_stack([safe_predict(m, f, X_missing) for m, f in base_models])
    #    y_pred = np.dot(test_preds, weights)

    #else:
    #   raise ValueError("Unknown ensemble method")
    return y_pred

def k_calibrate(CT,ozone_dosage,bromate_measured):
    def bromate_model(X, k, F):
        CT, ozone_dosage = X
        return k * CT + F * ozone_dosage
    popt, pcov = curve_fit(bromate_model, (CT, ozone_dosage), bromate_measured, p0=[0.1, 0.1],bounds=(0, np.inf))
    k_calibrated, F_calibrated = popt
    print(f"Calibrated k: {k_calibrated}, Calibrated F: {F_calibrated}")
    return k_calibrated, F_calibrated

def predict_bromate(new_ozone_dosage, new_CT, k, F):
    """
    Calculates predicted bromate outlet concentration.

    Parameters:
    new_ozone_dosage: Array or Series of new ozone doses (mg/L)
    new_CT: Array or Series of new CT values (mg*min/L)
    k: The calibrated kinetic constant
    F: The calibrated initial formation constant
    """
    # Apply the von Gunten-style linear empirical model
    prediction = (F * new_ozone_dosage) + (k * new_CT)

    # Physically, bromate cannot be negative.
    # If the model predicts a negative value due to noise, we clip it at 0.
    return np.maximum(0, prediction)

def physics_informed_bromate_mlp(X_train_ps, X_test_ps, y_train_ps, k_calibrated, F_calibrated,use_mlp):
    """
    Physics-informed MLP for bromate prediction.

    Physical relationship:
        Bromate_outlet = k * CT + F * Ozone_dosage

    If use_mlp=True → adds neural correction term.
    If use_mlp=False → uses pure physical model only.
    """

    # ----------------------------
    # Define the model architecture
    # ----------------------------
    class BromateMLP(nn.Module):
        def __init__(self, input_size, hidden_size, output_size, use_mlp=True):
            super(BromateMLP, self).__init__()
            self.use_mlp = use_mlp
            self.fc1 = nn.Linear(input_size, hidden_size)
            self.fc2 = nn.Linear(hidden_size, hidden_size)
            self.fc3 = nn.Linear(hidden_size, output_size)

            # Trainable physical constants
            self.k = nn.Parameter(torch.tensor(0.01))
            self.F = nn.Parameter(torch.tensor(0.01))

        def forward(self, x, CT, ozone_dose):
            if self.use_mlp:
                # MLP correction
                x = torch.tanh(self.fc1(x))
                x = torch.tanh(self.fc2(x))
                mlp_out = self.fc3(x)
            else:
                mlp_out = 0.0

            # Physical model component
            physics_out = self.k * CT + self.F * ozone_dose

            # Combine physics + residual
            return physics_out + mlp_out

    # ----------------------------
    # Convert data to tensors
    # ----------------------------
    X_train_tensor = torch.tensor(X_train_ps.values, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_ps.values, dtype=torch.float32).view(-1, 1)
    CT_train = X_train_ps['CT_All'].values
    ozone_train = X_train_ps['Ozonedosage'].values
    CT_train_tensor = torch.tensor(CT_train, dtype=torch.float32).view(-1, 1)
    ozone_train_tensor = torch.tensor(ozone_train, dtype=torch.float32).view(-1, 1)

    X_test_tensor = torch.tensor(X_test_ps.values, dtype=torch.float32)
    CT_test = X_test_ps['CT_All'].values
    ozone_test = X_test_ps['Ozonedosage'].values
    CT_test_tensor = torch.tensor(CT_test, dtype=torch.float32).view(-1, 1)
    ozone_test_tensor = torch.tensor(ozone_test, dtype=torch.float32).view(-1, 1)

    # ----------------------------
    # Initialize model
    # ----------------------------
    input_size = X_train_ps.shape[1]
    hidden_size = int(input("Set hidden layer size (e.g. 16, 32, 64): "))
    n_epochs = int(input("Set number of training epochs: "))
    lambda_data = float(input("Set weight for data loss (0-1): "))
    lambda_phys = float(input("Set weight for physics loss (0-1): "))
    model = BromateMLP(input_size, hidden_size, 1, use_mlp=use_mlp)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # ----------------------------
    # Training loop
    # ----------------------------
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()

        # Forward pass
        y_pred = model(X_train_tensor, CT_train_tensor, ozone_train_tensor)

        # Data loss
        loss_data = loss_fn(y_pred, y_train_tensor)

        # Physics loss: ensure prediction follows k * CT + F * ozone_dose
        physics_target = model.k * CT_train_tensor + model.F * ozone_train_tensor
        loss_phys = loss_fn(y_pred, physics_target)

        # Total loss
        loss = lambda_data * loss_data + lambda_phys * loss_phys
        loss.backward()
        optimizer.step()

        # Progress printout
        if epoch % 500 == 0 or epoch == n_epochs - 1:
            print(f"Epoch {epoch:5d} | Total Loss = {loss.item():.6f} | "
                  f"Data = {loss_data.item():.6f} | Phys = {loss_phys.item():.6f} | "
                  f"k = {model.k.item():.4f}, F = {model.F.item():.4f}")

    # ----------------------------
    # Predictions
    # ----------------------------
    model.eval()
    with torch.no_grad():
        y_pred_test = model(X_test_tensor, CT_test_tensor, ozone_test_tensor)

    y_pred_test_np = y_pred_test.numpy().flatten()

    print("\nFinal calibrated constants:")
    print(f"k = {model.k.item():.6f}")
    print(f"F = {model.F.item():.6f}")
    print(f"Use MLP correction: {use_mlp}")

    return model, model.k.item(), model.F.item(), y_pred_test_np

def plot_feature_importance_bromate(best_model, input_features, model_choice, X_train):
    """
    Plot feature importance and SHAP values for Random Forest, XGBoost,
    and ensemble models.

    Parameters:
    - best_model: trained model or list of trained models
    - input_features: list of feature names
    - model_choice: integer from 1 to 5
    - X: input DataFrame or array (used for SHAP)
    """
    results_dir ="Ensemble model 7 features - "
    os.makedirs(results_dir, exist_ok=True)

    def plot_importance_bar(importances, title):
        importance_df = pd.DataFrame({"Feature": input_features, "Importance": importances})
        importance_df = importance_df.sort_values(by="Importance", ascending=False)
        plt.figure(figsize=(10, 6))
        plt.barh(importance_df["Feature"], importance_df["Importance"], color='skyblue')
        plt.xlabel("Feature Importance Score")
        plt.ylabel("Features")
        plt.title(title)
        plt.gca().invert_yaxis()
        plt.savefig(os.path.join(results_dir, "Feauture importance.png"))
        plt.show()

    def plot_shap_summary(model, X_data, title):
        explainer = shap.Explainer(model, X_data)
        shap_values = explainer(X_data)
        shap.plots.beeswarm(shap_values, show=True, max_display=10)

    if model_choice == 1:  # Random Forest
        print("Random Forest - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "Random Forest - Feature Importance")
        print("Random Forest - SHAP Summary")
        plot_shap_summary(best_model, X_train, "Random Forest - SHAP Summary")

    elif model_choice == 2:  # XGBoost
        print("XGBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "XGBoost - Feature Importance")
        print("XGBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "XGBoost - SHAP Summary")

    elif model_choice == 3:  # GBoost
        print("GBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "GBoost - Feature Importance")
        print("GBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "GBoost - SHAP Summary")

    elif model_choice == 4:  # AdaBoost
        print("AdaBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "AdaBoost - Feature Importance")
        print("AdaBoost - SHAP Summary")
        background = shap.sample(X_train, 100).values
        X_sample = shap.sample(X_train, 100).values

        explainer = shap.KernelExplainer(best_model.predict, background)

        shap_values = explainer.shap_values(X_sample)

        shap.summary_plot(shap_values, X_sample, feature_names=input_features)

    elif model_choice == 5:  # LGBoost
        print("LGBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "LGBoost - Feature Importance")
        print("LGBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "LGBoost - SHAP Summary")

    else:
        raise ValueError("Feature importance is only supported for model choices 1 to 5.")

def predict_bromate_outlet(X, y, input_features, target_variable, k_calibrated, F_calibrated, models_info):
    """
    Main function to predict Bromate using user-selected model and inputs.
    """

    # Allow user to select a model
    print("\nSelect a model:")
    print("1. Random Forest")
    print("2. XGBoost")
    print("3. Gradient Boosting")
    print("4. AdaBoost")
    print("5. LightGBM")
    print("6. Physics-Informed MLP")
    print("7. van Gunten model")
    try:
        model_choice = int(input("Enter the number corresponding to your choice:"))
    except ValueError:
        print("Invalid choice. Exiting.")
        return

    # Train-test split
    testsize = float(input("Set the testing size(e.g. 0.1 = 10% for testing):"))
    rdmstate = int(input("Set the random state:"))
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X,y,range(len(X)), test_size=testsize, random_state=rdmstate) # this is for the decision trees approach

    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()

    X_train_scaled = feature_scaler.fit_transform(X_train)
    X_test_scaled = feature_scaler.transform(X_test)
    y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    scaler_dir = input('Set the folder name:').strip()
    full_save_dir = os.path.join(save_dir, scaler_dir)
    os.makedirs(full_save_dir, exist_ok=True)
    model_name1=f"{model_choice}feature_scaler.pkl"
    model_name2=f"{model_choice}target_scaler.pkl"
    save_path1=os.path.join(full_save_dir, model_name1)
    save_path2=os.path.join(full_save_dir, model_name2)
    joblib.dump(feature_scaler, save_path1)
    joblib.dump(target_scaler, save_path2)
    X_train = pd.DataFrame(X_train, columns=input_features)
    X_test = pd.DataFrame(X_test, columns=input_features)
    #X_train_ps, X_test_ps, y_train_ps, y_test_ps,idx_train, idx_test = train_test_split(X, y,range(len(X)), test_size=testsize, random_state=42)# this is for the PINN approach

    # Model selection
    best_model = None

    if model_choice == 1:
        model_type = "RF"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = random_forest_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 2:
        model_type = "XGB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = xgboost_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 3:
        model_type = "GB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = gb_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 4:
        model_type = "AdaB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = adaboost_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 5:
        model_type = "LGB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = lgb_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 6:
        model_type = "PINN - Physics"
        use_mlp=input("Use MLP correction or pure physics model? (y/n):")
        if use_mlp == "y":
            use_mlp = True
            model_type = "PINN - Physics + MLP"
        else:
            use_mlp = False
            model_type = "Physics only model"
        pinn, k_calibrated_val, F_calibrated_val,y_pred = physics_informed_bromate_mlp(X_train, X_test, y_train, k_calibrated, F_calibrated,use_mlp)
        best_model = pinn
        #y_pred = y_pred.to_numpy()
        y_pred = y_pred.reshape(-1, 1)
        y_test = y_test.values.reshape(-1, 1)
        #pinn, Ko3.item(), Kuva.item()

    elif model_choice == 7:
        model_type = "van Gunten"
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        ozonedosage=X_test['Ozonedosage']
        CTtest= X_test['CT_All']
        y_pred = predict_bromate(ozonedosage, CTtest, k_calibrated, F_calibrated)

    else:
        print("Invalid choice. Exiting.")
        return None

    # Calculate performance metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    si = rmse/np.mean(y_test)
    #kge = kling_gupta_efficiency(y_test, y_pred)

    print("\nModel Performance:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Scatter Index: {si:.4f}")
    print(f"R²: {r2:.4f}")
    #print(f"Kling Gupta Efficiency: {kge:.4f}")

    # Create results directory

    results_dir = f"{model_type} {X.shape[1]} features"
    os.makedirs(results_dir, exist_ok=True)

    # Save performance metrics to a txt file
    metrics_file = os.path.join(results_dir, "performance_metrics.txt")
    with open(metrics_file, "w") as f:
        f.write("Model Performance:\n")
        f.write(f"MAE: {mae:.4f}\n")
        f.write(f"MSE: {mse:.4f}\n")
        f.write(f"RMSE: {rmse:.4f}\n")
        f.write(f"Scatter Index: {si:.4f}\n")
        f.write(f"R²: {r2:.4f}\n")
        #f.write(f"Kling Gupta Efficiency: {kge:.4f}\n")

    # Save input features if available
    try:
        feature_file = os.path.join(results_dir, f"{X.shape[1]} used_features.txt")
        with open(feature_file, "w") as f:
            f.write(f"Number of Features: {X.shape[1]}\n")
            f.write("Input Features Used:\n")
            f.write("\n".join(X_test.columns))
    except:
        pass  # Skip if X_test doesn't have .columns

    # Save predicted vs observed plot
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, color='blue', alpha=0.7, label='Predictions')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='1:1 Line')
    plt.xlabel("Observed Bromate")
    plt.ylabel("Predicted Bromate")
    plt.title("Predicted vs Observed Bromate")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "predicted_vs_observed.png"))
    plt.show()

    # Plot predicted vs observed data
    #plt.scatter(y_test, y_pred, color='blue')
    #plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
    #plt.xlabel("Observed CT")
    #plt.ylabel("Predicted CT")
    #plt.title("Predicted vs Observed CT")
    #plt.show()

    # Plot feature importance if model is Random Forest or XGBoost
    if model_choice in [1, 2, 3, 4, 5]:
        plot_feature_importance_bromate(best_model, input_features,model_choice,X_train)

    return best_model, mae, mse, rmse, r2, si,y_test, y_pred

def kling_gupta_efficiency(y_test, y_pred):
    """Computes the Kling-Gupta Efficiency (KGE) metric."""
    y_true = np.asarray(y_test).flatten()
    y_predd = np.asarray(y_pred).flatten()
    eps = 1e-10   # numerical stability

    mean_obs = np.mean(y_true)
    mean_pred = np.mean(y_predd)

    std_obs = np.std(y_true)
    std_pred = np.std(y_predd)

    # Pearson correlation
    if std_obs < eps or std_pred < eps:
        r = 0.0
    else:
        r = np.corrcoef(y_true, y_predd)[0, 1]

    beta = mean_pred / (mean_obs +eps)  # Bias term

    cv_obs = std_obs / (mean_obs + eps)
    cv_pred = std_pred / (mean_pred + eps)
    gamma = cv_pred / (cv_obs + eps)

    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)

    print("\nModel Performance:")
    print(f"Std obs: {std_obs:.4f}")
    print(f"Std pred: {std_pred:.4f}")
    print(f"R crosscor: {r:.4f}")
    print(f"Beta: {beta:.4f}")
    print(f"gamma: {gamma:.4f}")
    print(f"Kling - Gupta - Efficiency (KGE): {kge:.4f}")
